# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
# YOUR CODE HERE: đọc data/splits/train.jsonl, xem vài mẫu, vẽ box lên ảnh
# 1a. Thống kê split + kiểm tra rò rỉ phiên quay
import os, sys, json, statistics
from collections import Counter
from pathlib import Path
sys.path.insert(0, os.getcwd())
from scripts.labcommon import CLASSES, read_jsonl

DATA = Path('data')
SPLITS = ('train', 'val', 'test')
rows = {s: read_jsonl(DATA / 'splits' / f'{s}.jsonl') for s in SPLITS}
meta = json.loads((DATA / 'splits' / 'meta.json').read_text(encoding='utf-8'))
print('dataset_version:', meta['dataset_version'])
print('mẫu 1 dòng train:', json.dumps(rows['train'][0], ensure_ascii=False)[:300], '\n')

# Số ảnh / phiên / box từng lớp mỗi split
print(f"{'split':6s}{'ảnh':>6s}{'phiên':>7s}{'box':>6s}" + ''.join(f'{c:>9s}' for c in CLASSES) + f"{'ảnh rỗng':>10s}")
for s in SPLITS:
    cnt = Counter(o['label'] for r in rows[s] for o in r['objects'])
    n_sess = len({r['session'] for r in rows[s]})
    empty = sum(not r['objects'] for r in rows[s])
    print(f"{s:6s}{len(rows[s]):6d}{n_sess:7d}{sum(cnt.values()):6d}"
          + ''.join(f'{cnt[c]:9d}' for c in CLASSES) + f'{empty:10d}')
    assert all(cnt[c] == meta['counts'][s]['boxes'][c] for c in CLASSES), s   # khớp meta.json

# Không phiên nào nằm ở hai split, không ảnh nào bị lặp
sess = {s: {r['session'] for r in rows[s]} for s in SPLITS}
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    common = sess[a] & sess[b]
    print(f'phiên chung {a:5s}–{b:5s}: {len(common)}')
    assert not common, common
ids = [r['image_id'] for s in SPLITS for r in rows[s]]
assert len(ids) == len(set(ids)), 'ảnh bị lặp giữa các split'

# Kích thước ảnh và kích thước box (diện tích tương đối so với khung hình)
print('\nkích thước ảnh:', dict(Counter((r['width'], r['height']) for s in SPLITS for r in rows[s])))
all_areas = []
for c in CLASSES:
    areas = [o['yolo'][2] * o['yolo'][3] for s in SPLITS for r in rows[s] for o in r['objects'] if o['label'] == c]
    all_areas += areas
    print(f'{c:8s} n={len(areas):5d}  diện tích trung vị = {100 * statistics.median(areas):.2f}% khung hình')
print(f'{"tất cả":8s} n={len(all_areas):5d}  diện tích trung vị = {100 * statistics.median(all_areas):.2f}% khung hình')

In [ ]:
# 1b. Vẽ 6 ảnh (2 ảnh/split) kèm box gốc từ nhãn YOLO → pixel
import random
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

COLORS = {'pothole': 'red', 'crack': 'yellow', 'manhole': 'cyan'}

def draw_gt(r):
    im = Image.open(DATA / r['image_path']).convert('RGB')
    d = ImageDraw.Draw(im)
    W, H = r['width'], r['height']
    for o in r['objects']:
        xc, yc, w, h = o['yolo']                       # chuẩn hoá [0,1], tâm + kích thước
        box = [(xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H]
        d.rectangle(box, outline=COLORS[o['label']], width=3)
        d.text((box[0] + 3, box[1] + 2), o['label'], fill=COLORS[o['label']])
    return im

rng = random.Random(0)
picks = []
for s in SPLITS:
    multi = [r for r in rows[s] if len({o['label'] for o in r['objects']}) >= 2]   # ưu tiên ảnh có ≥ 2 lớp
    pool = multi if len(multi) >= 2 else rows[s]
    picks += [(s, r) for r in rng.sample(pool, 2)]

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
for ax, (s, r) in zip(axes.flat, picks):
    ax.imshow(draw_gt(r)); ax.axis('off')
    ax.set_title(f"[{s}] {r['image_id']} · {r['session']} · {len(r['objects'])} box", fontsize=9)
fig.legend(handles=[plt.Line2D([], [], color=c, lw=3, label=l) for l, c in COLORS.items()],
           loc='lower center', ncol=3)
plt.tight_layout()
Path('report/figures').mkdir(parents=True, exist_ok=True)
fig.savefig('report/figures/cp1_gt_samples.jpg', dpi=80, bbox_inches='tight')
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# YOUR CODE HERE: viết yolo_to_box() và objects_to_answer() (GUIDE bước 2)

## 3. E0: zero-shot

In [ ]:
# YOUR CODE HERE: nạp model 4-bit, suy luận trên test → work/e0.jsonl (GUIDE bước 3)

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9